# Study Buddy V10.1 — Isolated GPU Tutor Prototype

This version keeps LiveTalking dependencies inside a separate virtual environment so Colab's shared Python packages are not overwritten.

In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## 1. Clone LiveTalking and create an isolated environment

The isolated environment prevents packages such as `websockets` from conflicting with Colab's own packages. LiveTalking currently pins `websockets==12.0`.

In [ ]:
%cd /content
!rm -rf LiveTalking
!git clone --depth 1 https://github.com/lipku/LiveTalking.git
!python -m venv --system-site-packages /content/livetalking-venv
!/content/livetalking-venv/bin/python -m pip install -q --upgrade pip
!/content/livetalking-venv/bin/python -m pip install -q -r /content/LiveTalking/requirements.txt
!/content/livetalking-venv/bin/python -c "import websockets; print('LiveTalking env websockets:', websockets.__version__)"
print('Isolated LiveTalking environment ready.')

## 2. Download the Wav2Lip test assets

LiveTalking documents `wav2lip256.pth` and the `wav2lip256_avatar1` archive as the quick-start assets.

In [ ]:
!/content/livetalking-venv/bin/python -m pip install -q gdown
!mkdir -p /content/livetalking_downloads
!gdown --folder 'https://drive.google.com/drive/folders/1FOC_MD6wdogyyX_7V1d4NDIO7P9NlSAJ?usp=sharing' -O /content/livetalking_downloads || true
!find /content/livetalking_downloads -maxdepth 3 -type f -printf '%p\n' | head -100

In [ ]:
import os, glob, shutil, tarfile
root='/content/LiveTalking'
download='/content/livetalking_downloads'
os.makedirs(f'{root}/models', exist_ok=True)
os.makedirs(f'{root}/data/avatars', exist_ok=True)
pths=glob.glob(download+'/**/wav2lip256.pth', recursive=True)
if pths:
    shutil.copy2(pths[0], f'{root}/models/wav2lip.pth')
    print('Copied Wav2Lip model.')
else:
    print('Upload wav2lip256.pth to /content/LiveTalking/models/wav2lip.pth if automatic download did not find it.')
archives=glob.glob(download+'/**/wav2lip256_avatar1.tar.gz', recursive=True)
if archives:
    with tarfile.open(archives[0], 'r:gz') as t:
        t.extractall(f'{root}/data/avatars')
    print('Extracted avatar archive.')
else:
    print('Upload/extract wav2lip256_avatar1.tar.gz into /content/LiveTalking/data/avatars/ if needed.')
print('Model present:', os.path.exists(f'{root}/models/wav2lip.pth'))
print('Avatar folders:', os.listdir(f'{root}/data/avatars')[:20])

## 3. Start LiveTalking

This proves the GPU avatar server can initialize. Colab is temporary and is not our final public WebRTC host.

In [ ]:
%cd /content/LiveTalking
!/content/livetalking-venv/bin/python app.py --transport webrtc --model wav2lip --avatar_id wav2lip256_avatar1 --max_session 1

## Next: Study Buddy V10.2

After the avatar server initializes, we connect browser microphone input, interruption detection, Groq, TTS, and the WebRTC avatar.